# Phase 7 — Prior Case Retrieval with the pre-trained **Transformer-XL + GQA** (decoder-only)

Given a query case and its split's candidate pool, return the 7 most relevant prior cases. The metric is **Micro-F1@7**. The Phase-1 Transformer-XL becomes a **bi-encoder**: every case is embedded independently, and relevance is a similarity between the embeddings. This notebook is the Transformer-XL twin of the GQLA and encoder–decoder Phase-7 notebooks: data, ranking, training, KD and submission cells are identical, and only the backbone differs.

```
case → windows <s> + 510 tokens + </s>  (8 spread over the judgment at test time; 4 random ones per case in training)
     → Transformer-XL (adapted)  → final hidden states of each window
     → attention pooling over tokens (zero-init = mean) → projection (identity init) → unit window vectors
     → attention aggregation over windows (zero-init = mean) → one unit vector per case
score(query, candidate) = cosine            (or "maxsim": window-level late interaction)
```

The untrained retriever is exactly mean-pooled Phase-1 Transformer-XL states, so `X0_zeroshot` measures transfer, and every method starts there.

**What is specific to Transformer-XL**
* **Leaner attention:** the attention uses the same maths as Phase 1 but less memory. `(q+u)·k` runs inside the fused SDPA kernel, and the relative-position term `shift((q+v)·r)` is passed as the additive mask. The q·k score matrix, softmax output and dropout mask are never stored. A self-test checks it against the original attention, with and without memory.
* **Prefix tuning through Transformer-XL's own `memory` slot:** each block gets learned memory, initialised from real layer inputs so it starts at the right scale.
* **KD student:** layer-strided, 8 → 4 blocks (`[0, 2, 5, 7]`), as in your Phase-3/4 Transformer-XL notebooks.
* **No segment recurrence:** windows are sampled across the case rather than contiguous, so they are encoded independently. Say so in the report.
* **Mixed precision:** fp16 autocast is checked against fp32 and switched off automatically if it is not finite and close.

**Training:** InfoNCE over the batch. Each query has 1 relevant case and 3 BM25 hard negatives, plus in-batch negatives, and relevant cases are never used as negatives. **GradCache** gives the exact full-batch gradient with the memory of one sub-batch, and a self-test verifies it against ordinary backprop.

**Techniques (the Phase-7 list):** `full` · `partial` · `lorma` (LoRMA⁺, compulsory) · `ia3` · `prefix` · `prompt` · `qlora` · `kd`, plus `lora` as an extra. BM25 over BPE ids serves as a lexical reference and as the source of hard negatives. An optional dense + BM25 *hybrid* is written as a separate `_hybrid` submission.

**Run order:** top to bottom. The checks cell runs the attention self-test, the fp16 check, PEFT identity at initialisation, and the GradCache gradient test. Then run BM25, zero-shot and the smoke test, and then the experiments.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os, sys, re, json, random, unicodedata, zipfile, time, math, pickle, gc, copy, shutil, glob
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # re-checked after loading

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token set:", bool(HF_TOKEN))

# ---- Phase-1 Transformer-XL + GQA artefacts (same paths as your Phase-4 TXL notebook; edit if yours differ) ----
P1_DIR = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
CHECKPOINT_PATH = os.path.join(P1_DIR, "transformer_xl_gqa_checkpoints", "best_model_transformer_xl_gqa-small.pt")
TOKENIZER_PATH = os.path.join(P1_DIR, "final_bpe_en.json")
MODEL_CONFIG = {"vocab_size": 32768, "d_model": 768, "num_layers": 8, "num_query_heads": 32, "num_kv_heads": 8,
                "memory_length": 128, "dropout": 0.1}

p7_dir = "/content/drive/MyDrive/Phase7_PCR_TXL"; os.makedirs(p7_dir, exist_ok=True)
local_ck = "/content/p7_ckpt"; os.makedirs(local_ck, exist_ok=True)

Mounted at /content/drive
device: cuda | Tesla T4
HF token set: True


In [ ]:
import os, re, json, heapq, collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PROC = None

def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0

WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
_WORKER = None
def _init_worker(tok):
    global _WORKER
    _WORKER = tok
def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]

class BPETokenizer:
    _HEAP_THRESHOLD = 24
    _CACHE_LIMIT = 500_000

    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = reserved_tokens if reserved_tokens is not None else ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}
        self.merges = {}
        self._ready = False
        self._cache = {}

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = BPETokenizer(vocab_size=data["vocab_size"], end_marker=data.get("end_marker", "</w>"),
                            reserved_tokens=data.get("reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]))
        tok.itos = data["itos"]; tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False; tok._cache = {}
        return tok

    def _prepare_inference(self):
        if self._ready: return
        self._byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]; self._eos = self.stoi["</s>"]
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError("Merge/vocab mismatch: cannot infer merge ids from ranks.")
        self._M = len(self.itos); M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}
        self._cache = {}; self._ready = True

    def _merge_small(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i
            if best is None: break
            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return ids

    def _merge_heap(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        n = len(ids)
        nxt = list(range(1, n + 1)); nxt[-1] = -1
        prv = list(range(-1, n - 1)); alive = [True] * n
        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None: heap.append((r, i))
        heapq.heapify(heap)
        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]: continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r: continue
            ids[i] = base + r; alive[j] = False
            k = nxt[j]; nxt[i] = k
            if k != -1: prv[k] = i
            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None: heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None: heapq.heappush(heap, (r2, i))
        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None: return hit
        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)
        ids = self._merge_small(ids) if len(ids) <= self._HEAP_THRESHOLD else self._merge_heap(ids)
        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT: self._cache[word] = result
        return result

    def encode(self, text, add_special_tokens=True):
        self._prepare_inference()
        out = [self._bos] if add_special_tokens else []
        extend = out.extend; cache_get = self._cache.get; bpe = self._bpe_word
        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None: t = bpe(w)
            extend(t)
        if add_special_tokens: out.append(self._eos)
        return out

    def encode_batch(self, texts, add_special_tokens=True, num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)
        if num_workers is None: num_workers = os.cpu_count() or 1
        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]
        import multiprocessing as mp
        chunks = [(texts[i:i + chunk_size], add_special_tokens) for i in range(0, len(texts), chunk_size)]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker, initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        self._prepare_inference()
        filtered_ids = [tid for tid in token_ids if self.itos[tid] not in self.reserved_tokens]
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()

if not os.path.isfile(TOKENIZER_PATH): raise FileNotFoundError(TOKENIZER_PATH)
tokenizer = BPETokenizer.from_json(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get("<pad>", 0); BOS_ID = tokenizer.stoi.get("<s>", 2); EOS_ID = tokenizer.stoi.get("</s>", 3)
VOCAB_SIZE = len(tokenizer.itos)
assert VOCAB_SIZE == MODEL_CONFIG["vocab_size"], "tokenizer vocab must match the Phase-1 checkpoint"
print("vocab", VOCAB_SIZE, "| PAD/BOS/EOS", PAD_ID, BOS_ID, EOS_ID)


def normalize_text(x):
    """the cleaning used by the Phase-4 Transformer-XL notebook (NFKC + whitespace collapse)"""
    s = unicodedata.normalize("NFKC", "" if x is None else str(x))
    return re.sub(r"\s+", " ", s.replace("\u00a0", " ")).strip()

vocab 32768 | PAD/BOS/EOS 0 2 3


In [ ]:
import math

class TransformerXLRelativePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        if d_model % 2:
            raise ValueError("d_model must be even")
        self.d_model = d_model
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, query_length: int, key_length: int, device=None):
        if key_length <= 0:
            raise ValueError("key_length must be positive")
        if device is None:
            device = self.inv_freq.device
        pos_seq = torch.arange(key_length - 1, -1, -1, dtype=torch.float32, device=device)
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq.to(device))
        return torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)

class TransformerXLGQAAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0):
        super().__init__()
        if d_model % num_query_heads != 0:
            raise ValueError("d_model must be divisible by num_query_heads")
        if num_query_heads % num_kv_heads != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_heads")
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.group_size = num_query_heads // num_kv_heads

        self.q_proj = nn.Linear(d_model, num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.r_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.u = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        self.v = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        nn.init.normal_(self.u, std=0.02)
        nn.init.normal_(self.v, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def _q_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)

    def _kv_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    def _expand_kv(self, x):
        B, H, T, D = x.shape
        return x.unsqueeze(2).expand(B, H, self.group_size, T, D).reshape(B, self.num_query_heads, T, D)

    @staticmethod
    def _rel_shift(x):
        B, H, Q, K = x.shape
        zero = torch.zeros((B, H, Q, 1), dtype=x.dtype, device=x.device)
        x = torch.cat([zero, x], dim=-1)
        x = x.view(B, H, K + 1, Q)
        x = x[:, :, 1:].view(B, H, Q, K)
        return x

    @staticmethod
    def _causal_mask(query_length, key_length, device):
        memory_length = key_length - query_length
        return torch.triu(torch.ones((query_length, key_length), dtype=torch.bool, device=device),
                           diagonal=memory_length + 1)

    def forward(self, x, memory=None, relative_position=None):
        B, Q, _ = x.shape
        if memory is not None and memory.size(1) > 0:
            kv_input = torch.cat([memory, x], dim=1)
        else:
            kv_input = x
        K = kv_input.size(1)

        q = self._q_heads(self.q_proj(x))
        k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
        v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))

        ac = torch.einsum("bhqd, bhkd->bhqk", q, k)
        ac = ac + torch.einsum("hd,bhkd->bhk", self.u, k).unsqueeze(2)

        if relative_position is None:
            raise ValueError("relative_position is required for Transformer-XL attention")

        r = self._expand_kv(self._kv_heads(self.r_proj(relative_position).unsqueeze(0)))
        bd = torch.einsum("bhqd,bhkd->bhqk", q, r)
        bd = bd + torch.einsum("hd,bhkd->bhk", self.v, r).unsqueeze(2)
        bd = self._rel_shift(bd)

        scores = (ac + bd) / math.sqrt(self.head_dim)
        mask = self._causal_mask(Q, K, x.device)
        scores = scores.masked_fill(mask.view(1, 1, Q, K), torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, Q, self.d_model)
        return self.out_proj(out)


class TransformerXLBlockGQA(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0, ffn_mult=4):
        super().__init__()
        self.attn = TransformerXLGQAAttention(d_model, num_query_heads, num_kv_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        ffn_dim = ffn_mult * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, memory, relative_position):
        x = self.norm1(x + self.attn(x, memory=memory, relative_position=relative_position))
        x = self.norm2(x + self.ffn(x))
        return x


class TransformerXLGQAModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_query_heads, num_kv_heads, memory_length,
                 dropout=0.0, tie_embeddings=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.memory_length = memory_length
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = TransformerXLRelativePositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerXLBlockGQA(d_model, num_query_heads, num_kv_heads, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, memory=None, labels=None):
        x = self.embedding(input_ids)
        Q = x.size(1)
        if memory is None:
            memory = [None] * self.num_layers
        if len(memory) != self.num_layers:
            raise ValueError(f"memory must contain {self.num_layers} entries")
        mem_len = 0 if memory[0] is None else memory[0].size(1)
        K = mem_len + Q
        rel_pos = self.pos_encoding(Q, K, device=x.device)
        for i, block in enumerate(self.blocks):
            x = block(x, memory[i], rel_pos)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), labels.reshape(-1))
        return {"logits": logits, "loss": loss}


if not os.path.isfile(CHECKPOINT_PATH): raise FileNotFoundError(CHECKPOINT_PATH)
ck = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state = ck.get("model_state_dict", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck
if isinstance(ck, dict): print("checkpoint keys:", list(ck.keys())[:12], "| epoch", ck.get("epoch"), "| best_val_loss", ck.get("best_val_loss"))
txl = TransformerXLGQAModel(**MODEL_CONFIG)
missing, unexpected = txl.load_state_dict(state, strict=False)
if missing or unexpected:                      # hard error on any mismatch (same rule as the Phase-3 notebook)
    print("Missing:", missing[:10], "count =", len(missing)); print("Unexpected:", unexpected[:10], "count =", len(unexpected))
    raise RuntimeError("Checkpoint is not compatible with MODEL_CONFIG / the Transformer-XL architecture above.")
del ck, state; gc.collect()
txl.eval()
print(f"Loaded Phase-1 Transformer-XL + GQA: {sum(p.numel() for n, p in txl.named_parameters() if not n.startswith('lm_head'))/1e6:.1f}M params (LM head is tied to the embedding and dropped below)")

checkpoint keys: ['epoch', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'best_val_loss', 'loss_history'] | epoch 7 | best_val_loss 3.077281383833674
Loaded Phase-1 Transformer-XL + GQA: 76.0M params (LM head is tied to the embedding and dropped below)


In [ ]:
# ============================================================
# 1. TRANSFORMER-XL ATTENTION, MEMORY-LEAN (replaces TransformerXLGQAAttention.forward; same weights, same maths)
#
# Original:  scores = [ q.k + u.k  +  shift((q.r) + v.r) ] / sqrt(d_h), causal mask, softmax, dropout, @ v
# Here:      (q + u).k is computed inside F.scaled_dot_product_attention (fused kernel, dropout passed to it), and the
#            relative-position term shift((q + v).r) / sqrt(d_h) (+ causal mask) is passed as the additive attn_mask.
#            Identical result; the q.k score matrix, the softmax output and the dropout mask are never stored, which
#            matters with 32 heads x 512 x 512 per chunk and many chunks per contrastive batch.
# The memory argument (Transformer-XL's segment cache) is unchanged: prefix tuning feeds learned "memory" through it.
# ============================================================
if not hasattr(TransformerXLGQAAttention, "_orig_forward"):
    TransformerXLGQAAttention._orig_forward = TransformerXLGQAAttention.forward


def txl_attn_forward(self, x, memory=None, relative_position=None):
    B, Q, _ = x.shape
    kv_input = torch.cat([memory.to(x.dtype), x], dim=1) if memory is not None and memory.size(1) > 0 else x
    K = kv_input.size(1)
    q = self._q_heads(self.q_proj(x))
    k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
    v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))
    r = self._expand_kv(self._kv_heads(self.r_proj(relative_position.to(x.dtype)).unsqueeze(0)))      # [1,H,K,dh]
    qv = q + self.v.to(q.dtype)[None, :, None, :]
    bd = self._rel_shift(torch.einsum("bhqd,hkd->bhqk", qv, r[0]))
    scale = 1.0 / math.sqrt(self.head_dim)
    mask = self._causal_mask(Q, K, x.device)
    bias = (bd * scale).masked_fill(mask.view(1, 1, Q, K), float("-inf")).to(q.dtype)
    qu = q + self.u.to(q.dtype)[None, :, None, :]
    drop = self.dropout.p if self.training else 0.0
    out = F.scaled_dot_product_attention(qu, k, v, attn_mask=bias, dropout_p=drop, scale=scale)
    return self.out_proj(out.transpose(1, 2).reshape(B, Q, self.d_model))


TransformerXLGQAAttention.forward = txl_attn_forward


class TXLEncoder(nn.Module):
    """Phase-1 Transformer-XL embedding + (selected) blocks + final norm; the LM head is dropped.
    prefix: optional learned per-layer memory [n_blocks, P, d] (prefix tuning through the XL memory slot)."""

    def __init__(self, g, keep=None):
        super().__init__()
        idx = list(range(len(g.blocks))) if keep is None else list(keep)
        self.keep = idx
        self.embed = copy.deepcopy(g.embedding)
        self.blocks = nn.ModuleList([copy.deepcopy(g.blocks[i]) for i in idx])
        self.final_norm = copy.deepcopy(g.final_norm)
        self.d_model = self.embed.embedding_dim
        self.pos = TransformerXLRelativePositionalEncoding(self.d_model)
        self.prefix = None


def txl_hidden(bb, ids=None, embeds=None, ckpt=False, return_layer_inputs=False):
    """right-padded chunks [N,L] (causal model: padding never reaches real tokens) -> final states [N,L,d]"""
    x = bb.embed(ids) if embeds is None else embeds
    N, Q = x.shape[:2]
    PL = 0 if bb.prefix is None else bb.prefix.size(1)
    rel = bb.pos(Q, Q + PL, device=x.device)
    lo = 0
    if torch.is_grad_enabled() and not x.requires_grad and PL == 0:
        while lo < len(bb.blocks) and not any(p.requires_grad for p in bb.blocks[lo].parameters()):
            lo += 1
    ins = []
    for i, blk in enumerate(bb.blocks):
        if return_layer_inputs:
            ins.append(x)
        m = bb.prefix[i].unsqueeze(0).expand(N, -1, -1) if PL else None
        if i < lo:
            with torch.no_grad():
                x = blk(x, m, rel)
        elif ckpt and torch.is_grad_enabled():
            x = grad_ckpt(lambda h, b=blk, m=m: b(h, m, rel), x, use_reentrant=False)
        else:
            x = blk(x, m, rel)
    h = bb.final_norm(x)
    return (h, ins) if return_layer_inputs else h


TXL_BASE = TXLEncoder(txl).eval()
del txl; gc.collect()
print(f"Transformer-XL encoder: {sum(p.numel() for p in TXL_BASE.parameters())/1e6:.1f}M params, "
      f"{len(TXL_BASE.blocks)} blocks, d={TXL_BASE.d_model}")

Transformer-XL encoder: 76.0M params, 8 blocks, d=768


In [ ]:
# ============================================================
# Small utilities: seeds, autocast, chunking, RNG capture (for GradCache), checkpoint I/O
# ============================================================
import collections


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(on=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(on and AMP_DTYPE is not None))


def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="uniform"):
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(strategy)


def get_rng():
    return (torch.get_rng_state(), torch.cuda.get_rng_state() if torch.cuda.is_available() else None)


def set_rng(st):
    torch.set_rng_state(st[0])
    if st[1] is not None:
        torch.cuda.set_rng_state(st[1])


def save_trainable(m, path, extra=None):
    torch.save({"trainable": {n: p.detach().cpu().clone() for n, p in m.named_parameters() if p.requires_grad},
                **(extra or {})}, path)


def load_trainable(m, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = [n for n, p in m.named_parameters() if p.requires_grad and n not in ck["trainable"]]
    if missing:
        raise RuntimeError(f"{path} lacks {len(missing)} trainable tensors, e.g. {missing[:3]}")
    m.load_state_dict(ck["trainable"], strict=False)
    return ck

In [ ]:
# ============================================================
# DATA: Phase-7 Prior Case Retrieval (query cases + one candidate pool per split)
#   Train/train.json + train_candidates.json | Dev/dev.json + dev_candidates.json | Test/test.json + test_candidates.json
#   queries: id, text (list of sentences), relevant_candidates (not in test) | candidates: id, text
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"
_REPO_FILES = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)


def find_repo_file(basename):
    hits = [f for f in _REPO_FILES if f.endswith("/" + basename) and ("pcr" in f.lower() or "phase_7" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 7, got {hits}. Phase-7-like files: "
                           f"{[f for f in _REPO_FILES if 'pcr' in f.lower()]}")
    return hits[0]


_TEXT_KEYS = ("text", "document", "judgment", "judgement", "sentences", "doc")
_REL_KEYS = ("relevant_candidates", "relevant_cases", "relevant", "positives")
_CAND_KEYS = ("candidates", "candidate_ids", "candidate_pool")    # per-query candidate list, if the data has one


def _as_text(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return " ".join(str(s).strip() for s in v if str(s).strip())
    return str(v).strip()


def _norm_id(x):
    return str(x).strip()


def _records(raw):
    if isinstance(raw, dict):                              # {id: {...}} or {id: text}
        return [({**v, "id": k} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    return raw


def read_json(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    return _records(json.load(open(path, encoding="utf-8")))


def read_queries(filename):
    raw = read_json(filename)
    keys = list(raw[0].keys())
    tk = next(k for k in _TEXT_KEYS if k in keys)
    rk = next((k for k in _REL_KEYS if k in keys), None)
    ck = next((k for k in _CAND_KEYS if k in keys and isinstance(raw[0][k], list)), None)
    print(f"{filename}: {len(raw)} queries | fields {keys} -> text '{tk}', relevance '{rk}', per-query candidates '{ck}'")
    if rk is None and not filename.startswith("test"):
        raise KeyError(f"no relevance field in {filename} among {_REL_KEYS}: add its name from the fields above")
    return [dict(id=_norm_id(r["id"]), text=_as_text(r[tk]),
                 rel=None if rk is None else [_norm_id(c) for c in (r.get(rk) or [])],
                 cands=None if ck is None else [_norm_id(c) for c in r[ck]]) for r in raw]


def read_pool(filename):
    raw = read_json(filename)
    keys = list(raw[0].keys())
    tk = next(k for k in _TEXT_KEYS if k in keys)
    print(f"{filename}: {len(raw)} candidates | fields {keys} -> text '{tk}'")
    pool = {}
    for r in raw:
        pool[_norm_id(r["id"])] = _as_text(r[tk])
    return pool


SPL = {}
for split, d in (("train", "train"), ("dev", "dev"), ("test", "test")):
    SPL[split] = dict(queries=read_queries(f"{d}.json"), pool=read_pool(f"{d}_candidates.json"))

# ---- sanity + statistics ---------------------------------------------------------------------------------------
for split in ("train", "dev", "test"):
    qs, pool = SPL[split]["queries"], SPL[split]["pool"]
    self_in_pool = sum(q["id"] in pool for q in qs)
    msg = f"[{split}] {len(qs)} queries | pool {len(pool)} | queries that are also in their pool: {self_in_pool}"
    if qs[0]["rel"] is not None:
        n_rel = np.array([len(q["rel"]) for q in qs])
        missing = sum(c not in pool for q in qs for c in q["rel"])
        msg += (f" | relevant/query: mean {n_rel.mean():.2f} median {np.median(n_rel):.0f} max {n_rel.max()} "
                f"| zero-relevant queries {(n_rel == 0).sum()} | relevant ids missing from pool {missing}")
        best_f1 = 2 * np.minimum(n_rel, 7).sum() / (7 * len(qs) + n_rel.sum())
        msg += f" | oracle Micro-F1@7 {best_f1:.3f}"
    print(msg)
print("pool overlap train/dev:", len(set(SPL["train"]["pool"]) & set(SPL["dev"]["pool"])),
      "| train/test:", len(set(SPL["train"]["pool"]) & set(SPL["test"]["pool"])))
_rel_cnt = collections.Counter(c for q in SPL["train"]["queries"] for c in q["rel"])
print("train: most-cited candidates are relevant to", [n for _, n in _rel_cnt.most_common(5)], "queries")


# ---- tokenise every unique document once (cached on Drive) ------------------------------------------------------
def _tok_cached(texts, tag, bs=500, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p7_dir, f"tok_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(texts), bs):
        part = [normalize_text(t) for t in texts[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(part, add_special_tokens=False, num_workers=workers, chunk_size=8)]
        print(f"  tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


TOK = {}                    # (split, "q"|"c", id) -> int32 token array
for split in ("train", "dev", "test"):
    qs, pool = SPL[split]["queries"], SPL[split]["pool"]
    qt = _tok_cached([q["text"] for q in qs], f"{split}_queries")
    pids = list(pool)
    ct = _tok_cached([pool[i] for i in pids], f"{split}_pool")
    for q, t in zip(qs, qt):
        TOK[(split, "q", q["id"])] = t
    for i, t in zip(pids, ct):
        TOK[(split, "c", i)] = t
    SPL[split]["pool_ids"] = pids
    SPL[split]["qids"] = [q["id"] for q in qs]
    SPL[split]["rel"] = {q["id"]: set(q["rel"]) & set(pool) for q in qs} if qs[0]["rel"] is not None else None
    # allowed candidates per query: the per-query list if the data has one, else the whole split pool
    SPL[split]["allowed"] = ({q["id"]: [c for c in q["cands"] if c in pool] for q in qs} if qs[0]["cands"] is not None
                             else None)

_lens = np.array([len(TOK[("train", "c", i)]) for i in SPL["train"]["pool_ids"]])
_qlens = np.array([len(TOK[("train", "q", i)]) for i in SPL["train"]["qids"]])
pct = lambda a: "  ".join(f"p{p} {int(np.percentile(a, p))}" for p in (10, 50, 90, 99))
print("train candidate BPE tokens:", pct(_lens), f"| 512-chunks median {np.median(np.ceil(_lens / 511)):.0f}")
print("train query     BPE tokens:", pct(_qlens))

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train.json: 827 queries | fields ['text', 'relevant_candidates', 'id'] -> text 'text', relevance 'relevant_candidates', per-query candidates 'None'


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train_candidates.json: 4320 candidates | fields ['text', 'id'] -> text 'text'


... (6 outputs left out)

test.json: 237 queries | fields ['text', 'id'] -> text 'text', relevance 'None', per-query candidates 'None'


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test_candidates.json: 1727 candidates | fields ['text', 'id'] -> text 'text'
[train] 827 queries | pool 4320 | queries that are also in their pool: 827 | relevant/query: mean 6.42 median 4 max 52 | zero-relevant queries 10 | relevant ids missing from pool 0 | oracle Micro-F1@7 0.639
[dev] 118 queries | pool 1023 | queries that are also in their pool: 118 | relevant/query: mean 8.25 median 5 max 92 | zero-relevant queries 0 | relevant ids missing from pool 0 | oracle Micro-F1@7 0.618
[test] 237 queries | pool 1727 | queries that are also in their pool: 237
pool overlap train/dev: 436 | train/test: 740
train: most-cited candidates are relevant to [14, 10, 10, 9, 8] queries
train candidate BPE tokens: p10 1200  p50 4575  p90 15464  p99 60872 | 512-chunks median 9
train query     BPE tokens: p10 1983  p50 5714  p90 15708  p99 55126


In [ ]:
# ============================================================
# Ranking utilities, Micro-F1@7, submission writer, and a BM25 reference ranker over BPE ids
#
# BM25 is NOT a pretrained model: it is a term-weighting formula over our own tokenizer's ids. It is used for
#   (a) a lexical reference point for the report, (b) hard-negative mining, (c) optional dense+lexical fusion.
# Every submitted system still ranks with the fine-tuned Phase-1 model; fusion is reported separately.
# ============================================================
import scipy.sparse as sp
import zipfile

TOPK = 7


def valid_mask(split):
    """[nq, nc] bool: candidate allowed for that query (same split pool, per-query list if any, never itself)"""
    qids, pids = SPL[split]["qids"], SPL[split]["pool_ids"]
    col = {p: j for j, p in enumerate(pids)}
    allowed = SPL[split].get("allowed")
    if allowed is None:
        m = np.ones((len(qids), len(pids)), dtype=bool)
    else:
        m = np.zeros((len(qids), len(pids)), dtype=bool)
        for i, q in enumerate(qids):
            m[i, [col[c] for c in allowed[q]]] = True
    for i, q in enumerate(qids):
        if q in col:
            m[i, col[q]] = False
    return m


def rank_ids(S, split, k=TOPK):
    """S [nq, nc] scores (numpy) -> {qid: [top-k candidate ids]} (invalid candidates never returned)"""
    S = np.where(valid_mask(split), S, -np.inf)
    k_eff = min(k, S.shape[1])
    top = np.argpartition(-S, k_eff - 1, axis=1)[:, :k_eff]
    order = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, 1), axis=1, kind="stable"), 1)
    pids, qids = SPL[split]["pool_ids"], SPL[split]["qids"]
    return {q: [pids[j] for j in order[i] if np.isfinite(S[i, j])] for i, q in enumerate(qids)}


def micro_at_k(pred, rel, k=TOPK):
    hits = n_pred = n_gold = 0
    for q, gold in rel.items():
        p = pred.get(q, [])[:k]
        hits += len(set(p) & gold); n_pred += k; n_gold += len(gold)
    P = hits / max(n_pred, 1); R = hits / max(n_gold, 1)
    return dict(f1=0.0 if P + R == 0 else 2 * P * R / (P + R), p=P, r=R)


def full_metrics(S, split):
    """Micro-P/R/F1@7 plus Recall@50/100 and MRR (for analysis)"""
    rel = SPL[split]["rel"]
    m = micro_at_k(rank_ids(S, split, TOPK), rel)
    Sm = np.where(valid_mask(split), S, -np.inf)
    order = np.argsort(-Sm, axis=1, kind="stable")[:, :100]
    pids = SPL[split]["pool_ids"]
    r50 = r100 = mrr = 0.0; n = 0
    for i, q in enumerate(SPL[split]["qids"]):
        g = rel[q]
        if not g:
            continue
        ids = [pids[j] for j in order[i]]
        r50 += len(set(ids[:50]) & g) / len(g); r100 += len(set(ids) & g) / len(g)
        mrr += next((1 / (r + 1) for r, c in enumerate(ids) if c in g), 0.0); n += 1
    m.update(r50=r50 / max(n, 1), r100=r100 / max(n, 1), mrr=mrr / max(n, 1))
    return m


def fmt_m(m):
    return (f"F1@7 {m['f1']:.4f} (P {m['p']:.4f} R {m['r']:.4f})"
            + (f" | R@50 {m['r50']:.3f} R@100 {m['r100']:.3f} MRR {m['mrr']:.3f}" if "r50" in m else ""))


def write_submission(S, name, split="test"):
    pred = rank_ids(S, split, TOPK)
    pool = set(SPL[split]["pool_ids"])
    for q in SPL[split]["qids"]:
        p = pred[q]
        assert len(p) == TOPK and len(set(p)) == TOPK and all(c in pool for c in p), f"bad prediction for {q}: {p}"
    d = os.path.join(p7_dir, f"submission_{name}"); os.makedirs(d, exist_ok=True)
    jp = os.path.join(d, "prediction.json")
    json.dump(pred, open(jp, "w"), indent=1)
    zp = os.path.join(d, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    print(f"wrote {len(pred)} queries x {TOPK} ids -> {zp}")
    return zp


# ---------------- BM25 over BPE ids (unigrams + hashed bigrams) ----------------
V_TOK = len(tokenizer.itos)
_HASH = 1 << 22


def _terms(ids, ngram):
    ids = np.asarray(ids, dtype=np.int64)
    t = [ids]
    if ngram >= 2 and len(ids) > 1:
        t.append(V_TOK + (ids[:-1] * 1000003 + ids[1:]) % _HASH)
    return np.unique(np.concatenate(t), return_counts=True)


def _count_matrix(docs, ngram):
    rows, cols, vals = [], [], []
    for i, d in enumerate(docs):
        u, c = _terms(d, ngram)
        rows.append(np.full(len(u), i)); cols.append(u); vals.append(c)
    return sp.csr_matrix((np.concatenate(vals).astype(np.float32), (np.concatenate(rows), np.concatenate(cols))),
                         shape=(len(docs), V_TOK + _HASH))


def bm25_scores(split, k1=1.2, b=0.75, ngram=2, max_df=0.5, query_tf=False):
    """-> [nq, nc] float32 BM25 scores of the split's queries against the split's pool"""
    key = (split, k1, b, ngram, max_df, query_tf)
    if key in _BM25:
        return _BM25[key]
    C = _count_matrix([TOK[(split, "c", i)] for i in SPL[split]["pool_ids"]], ngram)
    Q = _count_matrix([TOK[(split, "q", i)] for i in SPL[split]["qids"]], ngram)
    N = C.shape[0]
    df = np.asarray((C > 0).sum(0)).ravel()
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    idf[df > max_df * N] = 0.0                                   # stop-word-like pieces
    dl = np.asarray(C.sum(1)).ravel(); avg = dl.mean()
    C = C.tocoo()
    denom = C.data + k1 * (1 - b + b * dl[C.row] / avg)
    W = sp.csr_matrix((idf[C.col] * C.data * (k1 + 1) / denom, (C.row, C.col)), shape=C.shape)
    Qw = Q.copy()
    if not query_tf:
        Qw.data[:] = 1.0
    else:
        Qw.data = Qw.data / (Qw.data + 1.2) * 2.2
    S = (Qw @ W.T).toarray().astype(np.float32)
    _BM25[key] = S
    return S


_BM25 = {}


def zrow(S):
    """per-query z-normalisation over valid candidates (for score fusion)"""
    S = S.astype(np.float64)
    mu, sd = S.mean(1, keepdims=True), S.std(1, keepdims=True) + 1e-9
    return (S - mu) / sd


def tune_fusion(S_dense, split="dev", lex=None, grid=np.linspace(0, 1, 11)):
    """w * z(dense) + (1 - w) * z(BM25): best w on the labelled split"""
    lex = bm25_scores(split) if lex is None else lex
    zd, zl = zrow(S_dense), zrow(lex)
    res = [(w, micro_at_k(rank_ids(w * zd + (1 - w) * zl, split), SPL[split]["rel"])["f1"]) for w in grid]
    return max(res, key=lambda x: x[1]), res

In [ ]:
# ============================================================
# Parameter-efficient adaptation modules (own implementations, no peft library)
#   LoRA     (Hu et al., 2021)            W x + (a/r) B A x                      B = 0 at init
#   LoRMA+   (Bihany, Patel, Modi, 2025)   (I + (a/r) B A) W0 x                   multiplicative, additive rank
#                                          inflation; B = 0 at init -> identity; computed as y0 + s*B(A y0),
#                                          y0 = W0 x, so no d x d matrix is ever formed (the paper's re-ordering)
#   QLoRA    (Dettmers et al., 2023)       frozen base stored as 4-bit NF4 (blockwise absmax), LoRA on top
#   Adapters (Houlsby et al., 2019 / Pfeiffer et al., 2021)  bottleneck MLP, zero-init up-projection
#   IA3      (Liu et al., 2022)            learned rescaling vectors: keys, values and the FFN inner activation;
#                                          init 1 -> identity; ~0.1% extra parameters
#   Prefix   (Li & Liang, 2021)            learned per-layer key/value prefixes (built in the model code)
#   Prompt   (Lester et al., 2021)         learned soft input tokens (built in the model code)
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """LoRMA+ : W = (I_out + (a/r) B A) W0 ;  B in R^{out x r}, A in R^{r x out}. The bias is added after the
    multiplicative transform (it is not part of W0)."""

    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64.
    Dequantised on the fly in forward (memory saving of QLoRA, without bitsandbytes)."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):                       # chunked argmin keeps memory small
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class IA3Linear(nn.Module):
    """IA3: where='out' -> l * W x (keys / values) ; where='in' -> W (l * x) (FFN inner activation, i.e. the input of
    the FFN output projection). l is initialised to 1, so the model starts unchanged."""

    def __init__(self, base, where="out"):
        super().__init__()
        self.base, self.where = base, where
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.l = nn.Parameter(torch.ones(base.out_features if where == "out" else base.in_features))

    def forward(self, x):
        if self.where == "in":
            return self.base(x * self.l.to(x.dtype))
        return self.base(x) * self.l.to(x.dtype)


class Adapter(nn.Module):
    """bottleneck adapter: x + up(act(down(x))); up is zero-initialised so the model starts unchanged"""

    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """Applies an adapter to the output of a sub-layer (Houlsby: after attention and FFN; Pfeiffer: after FFN).
    Works for sub-layers that return a tensor or a tuple (tensor, cache)."""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def replace_linears(root, make, keep):
    """Replace every nn.Linear child c (full name n) under root with make(c) when keep(n) is True."""
    done = []
    for name, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            full = f"{name}.{cname}" if name else cname
            if type(child) is nn.Linear and keep(full):
                setattr(mod, cname, make(child)); done.append(full)
    return done


def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

In [ ]:
# ============================================================
# 2. TRANSFORMER-XL AS A CASE ENCODER + ADAPTATION METHODS
#
# Chunk format = the Phase-4 window format: <s> + 510 tokens + </s>. Chunks are encoded independently; the segment
# recurrence (memory) is not used to link chunks (chunks are sampled across the case, not contiguous), and the memory
# slot is used only by prefix tuning. Methods (the Phase-7 list):
#   full | partial (top-k blocks) | lorma (LoRMA+) | qlora (NF4 base + LoRA) | ia3 | prefix | prompt | kd
#   (+ lora as an extra, zeroshot = no training). r_proj (positional projection) and u / v stay frozen.
# ============================================================
D_MODEL = TXL_BASE.d_model
CHUNK_BODY = 510
MODEL_CFG_OVERRIDES = dict(student_layers=4)              # KD student: 4 layer-strided blocks [0, 2, 5, 7]


def finish_chunk(c):
    return np.concatenate([[BOS_ID], np.asarray(c, dtype=np.int64), [EOS_ID]])


def backbone_embedding(bb):
    return bb.embed


def backbone_encode(bb, ids, am, prompt=None, ckpt=False):
    if prompt is None:
        return txl_hidden(bb, ids, ckpt=ckpt), am
    x = bb.embed(ids)
    P = prompt.size(0)
    x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], 1)
    return txl_hidden(bb, embeds=x, ckpt=ckpt), torch.cat([am.new_ones(am.size(0), P), am], 1)


TXL_TARGETS = ("attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj", "ffn.0", "ffn.3")


def _is_target(name):
    return name.startswith("blocks.") and any(name.endswith(t) for t in TXL_TARGETS)


@torch.no_grad()
def _real_memory(bb, P):
    """layer inputs at P random positions of a real training chunk: prefix-memory init in the right scale"""
    ids = torch.as_tensor(finish_chunk(TOK[("train", "c", SPL["train"]["pool_ids"][0])][:CHUNK_BODY]))[None]
    _, ins = txl_hidden(bb.to(device).eval(), ids.to(device), return_layer_inputs=True)
    pos = torch.randperm(ids.size(1))[:P].sort().values
    return torch.stack([h[0, pos.to(h.device)] for h in ins]).float().cpu()          # [n_blocks, P, d]


def apply_method(bb, cfg):
    meth = cfg.method
    freeze(bb)
    if meth == "zeroshot":
        return "Phase-1 Transformer-XL as is (no training)"
    if meth == "full":
        for p in bb.parameters():
            p.requires_grad = True
        return "all Transformer-XL weights"
    if meth == "partial":
        for blk in bb.blocks[len(bb.blocks) - cfg.unfreeze_top_k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in bb.final_norm.parameters():
            p.requires_grad = True
        return f"top {cfg.unfreeze_top_k} blocks + final norm"
    if meth == "lorma":
        n = replace_linears(bb, lambda l: LoRMALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRMA+ r={cfg.r} on {len(n)} linears"
    if meth == "lora":
        n = replace_linears(bb, lambda l: LoRALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRA r={cfg.r} on {len(n)} linears"
    if meth == "qlora":
        n = replace_linears(bb, lambda l: LoRALinear(NF4Linear(l), cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"QLoRA (NF4 base) r={cfg.r} on {len(n)} linears"
    if meth == "ia3":
        for blk in bb.blocks:
            blk.attn.k_proj = IA3Linear(blk.attn.k_proj, "out")
            blk.attn.v_proj = IA3Linear(blk.attn.v_proj, "out")
            blk.ffn[3] = IA3Linear(blk.ffn[3], "in")                  # scales the FFN hidden activation
        return f"IA3 on k_proj, v_proj and FFN hidden: {3 * len(bb.blocks)} vectors"
    if meth == "prefix":
        bb.prefix = nn.Parameter(_real_memory(bb, cfg.prefix_len))
        return f"prefix tuning: {cfg.prefix_len} learned memory slots per block (Transformer-XL memory, real-state init)"
    if meth == "prompt":
        return f"prompt tuning, {cfg.prompt_len} soft tokens before every chunk"
    raise ValueError(f"unknown method {meth!r}")


def build_backbone(cfg):
    bb = copy.deepcopy(TXL_BASE)
    pre = {id(p) for p in bb.parameters()}
    desc = apply_method(bb, cfg)
    return bb, desc, pre


def make_student_backbone(teacher_bb, cfg):
    """layer-strided student from the fine-tuned teacher (8 -> 4 keeps blocks [0, 2, 5, 7]; teacher = full/partial run)"""
    n = cfg.student_layers
    keep = np.linspace(0, len(teacher_bb.blocks) - 1, n).round().astype(int).tolist()
    s = copy.deepcopy(teacher_bb)
    s.blocks = nn.ModuleList([s.blocks[i] for i in keep]); s.keep = keep; s.prefix = None
    print(f"student: {n} blocks from teacher blocks {keep} | {sum(p.numel() for p in s.parameters())/1e6:.1f}M params "
          f"(teacher {sum(p.numel() for p in teacher_bb.parameters())/1e6:.1f}M)")
    return s


# ---------------- self-test: lean attention == Phase-1 attention (with and without memory); fp16 safety ----------------
@torch.no_grad()
def check_backbone():
    global AMP_DTYPE
    bb = TXL_BASE.to(device).eval()
    docs = [TOK[("train", "c", i)] for i in SPL["train"]["pool_ids"][:50] if len(TOK[("train", "c", i)]) >= 300][:4]
    ids = torch.as_tensor(np.stack([finish_chunk(d[:254]) for d in docs]), dtype=torch.long, device=device)
    for mem in (None, torch.randn(len(bb.blocks), 16, D_MODEL, device=device)):
        bb.prefix = mem
        TransformerXLGQAAttention.forward = TransformerXLGQAAttention._orig_forward
        ref = txl_hidden(bb, ids)
        TransformerXLGQAAttention.forward = txl_attn_forward
        new = txl_hidden(bb, ids)
        d = (ref - new).abs().max().item()
        print(f"lean vs Phase-1 attention ({'with 16 memory slots' if mem is not None else 'no memory'}): max|diff| {d:.2e}")
        assert d < 1e-3 * max(1.0, ref.abs().max().item()), "patched Transformer-XL attention differs"
    bb.prefix = None
    if AMP_DTYPE is not None:
        with amp_ctx():
            h16 = txl_hidden(bb, ids)
        ok = bool(torch.isfinite(h16).all())
        cos = F.cosine_similarity(h16.float().reshape(-1, D_MODEL), new.reshape(-1, D_MODEL), -1).min().item() if ok else 0.0
        print(f"{AMP_DTYPE} autocast: finite={ok}, min cosine to fp32 {cos:.4f}")
        if not ok or cos < 0.98:
            AMP_DTYPE = None
            print("-> mixed precision switched OFF (fp32 everywhere)")

In [ ]:
# ============================================================
# RETRIEVER: Phase-1 backbone -> chunk vectors -> document representation -> similarity
#
#   document -> 512-token chunks (K spread over the case; K_train sampled at random during training = augmentation)
#            -> backbone (adapted with the chosen method) -> token pooling (attention pooling, zero-init = mean)
#            -> projection d->d (identity init) -> L2-normalised CHUNK vectors
#   score_mode "single": chunk vectors -> attention aggregation (zero-init = mean) -> one vector per case -> cosine
#   score_mode "maxsim": late interaction at chunk level: mean over query chunks of max cosine over candidate chunks
#
# With the identity / zero initialisations, the untrained retriever IS mean-pooled Phase-1 embeddings, so every
# experiment starts from the zero-shot transfer baseline (method="zeroshot" evaluates exactly that).
# Model-specific hooks (defined in the backbone cell): CHUNK_BODY, finish_chunk(), backbone_encode(), D_MODEL.
# ============================================================
class Retriever(nn.Module):
    def __init__(self, bb, cfg):
        super().__init__()
        self.bb, self.cfg = bb, cfg
        d = D_MODEL
        self.tok_w = nn.Parameter(torch.zeros(d))                  # token attention pooling (zero -> mean)
        self.proj = nn.Linear(d, d)
        with torch.no_grad():
            self.proj.weight.copy_(torch.eye(d)); self.proj.bias.zero_()
        self.chunk_w = nn.Parameter(torch.zeros(d))                # chunk aggregation (zero -> mean)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / cfg.tau)))
        self.prompt = None
        if cfg.method == "prompt":
            emb = backbone_embedding(bb).weight.detach()
            self.prompt = nn.Parameter(emb[torch.randint(1000, 20000, (cfg.prompt_len,))].clone())

    def head_param_ids(self):
        return {id(p) for p in (self.tok_w, self.proj.weight, self.proj.bias, self.chunk_w, self.logit_scale)}

    # ---- chunks -> unit vectors ---------------------------------------------------------------
    def chunk_vec(self, ids, am):
        H, m = backbone_encode(self.bb, ids, am, self.prompt, ckpt=self.cfg.grad_ckpt and self.training)
        m = m.clone()
        if self.prompt is not None:
            m[:, : self.prompt.size(0)] = False                    # pool over the text, not the soft prompt
        H = H.float()
        pool = self.cfg.pool
        if pool == "last":
            last = m.long().cumsum(1).argmax(1)
            v = H[torch.arange(H.size(0), device=H.device), last]
        else:
            logit = (H @ self.tok_w) if pool == "attn" else torch.zeros(m.shape, device=H.device)
            a = torch.softmax(logit.masked_fill(~m, -1e4), dim=1)
            v = (a.unsqueeze(-1) * H).sum(1)
        return F.normalize(self.proj(v), dim=-1)

    # ---- chunk vectors -> per-document representations -------------------------------------------
    def doc_reps(self, V, doc, slot, n_docs):
        K = int(slot.max()) + 1
        mat = V.new_zeros(n_docs, K, V.size(1)).index_put((doc, slot), V)
        msk = torch.zeros(n_docs, K, dtype=torch.bool, device=V.device).index_put((doc, slot), torch.ones_like(doc, dtype=torch.bool))
        a = torch.softmax((mat @ self.chunk_w).masked_fill(~msk, -1e4), dim=1)
        single = F.normalize((a.unsqueeze(-1) * mat).sum(1), dim=-1)
        return single, mat, msk

    def scale(self):
        return self.logit_scale.clamp(max=math.log(100.0)).exp()


def score_reps(q, c, mode, block=64):
    """q, c: (single [n,d], mat [n,K,d], mask [n,K]) -> cosine-type scores [nq, nc] (unscaled)"""
    if mode == "single":
        return q[0] @ c[0].T
    out = []
    cm, cmask = c[1], c[2]
    for s in range(0, q[1].size(0), block):
        qm, qmask = q[1][s:s + block], q[2][s:s + block]
        sim = torch.einsum("bkd,nld->bnkl", qm, cm)
        sim = sim.masked_fill(~cmask[None, :, None, :], -1e4).amax(-1)            # best candidate chunk
        qmf = qmask[:, None, :].to(sim.dtype)
        out.append((sim * qmf).sum(-1) / qmf.sum(-1).clamp_min(1))               # mean over query chunks
    return torch.cat(out, 0)


# ---------------- chunk layout for a list of documents ----------------
def doc_chunk_list(tokens, K, strategy="uniform", rng=None):
    ch = chunk_ids(tokens, CHUNK_BODY, CHUNK_BODY)
    if rng is not None and len(ch) > K:
        ch = [ch[i] for i in sorted(rng.choice(len(ch), K, replace=False))]
    else:
        ch = select_chunks(ch, K, strategy)
    return [finish_chunk(c) for c in ch]


def layout(docs_chunks):
    flat, doc, slot = [], [], []
    for i, chs in enumerate(docs_chunks):
        for j, c in enumerate(chs):
            flat.append(c); doc.append(i); slot.append(j)
    return flat, torch.tensor(doc, device=device), torch.tensor(slot, device=device)


def pad_batch(chunks):
    T = max(max(len(c), 1) for c in chunks)
    ids = torch.full((len(chunks), T), PAD_ID, dtype=torch.long)
    am = torch.zeros(len(chunks), T, dtype=torch.bool)
    for i, c in enumerate(chunks):
        if len(c):
            ids[i, :len(c)] = torch.from_numpy(np.asarray(c, dtype=np.int64)); am[i, :len(c)] = True
        else:
            am[i, 0] = True
    return ids.to(device), am.to(device)


def sub_batches(flat, max_chunks, tok_budget):
    """length-sorted sub-batches bounded by chunk count and token budget"""
    order = sorted(range(len(flat)), key=lambda i: len(flat[i]))
    out, cur = [], []
    for i in order:
        if cur and (len(cur) >= max_chunks or (len(cur) + 1) * max(len(flat[i]), 1) > tok_budget):
            out.append(cur); cur = []
        cur.append(i)
    if cur:
        out.append(cur)
    return out


# ---------------- encoding a whole split (evaluation / caching) ----------------
@torch.no_grad()
def encode_docs(rt, keys, K, strategy="uniform", bs=64, tok_budget=24576):
    """keys: list of TOK keys -> (single [n,d], mat [n,K,d], mask [n,K]) on the GPU (fp16 for the big ones)"""
    rt.eval()
    single, mats, masks = [], [], []
    for s in range(0, len(keys), 256):
        part = keys[s:s + 256]
        flat, doc, slot = layout([doc_chunk_list(TOK[k], K, strategy) for k in part])
        V = torch.empty(len(flat), D_MODEL, device=device)
        for sb in sub_batches(flat, bs, tok_budget):
            ids, am = pad_batch([flat[i] for i in sb])
            with amp_ctx():
                V[torch.tensor(sb, device=device)] = rt.chunk_vec(ids, am).float()
        sg, mt, mk = rt.doc_reps(V, doc, slot, len(part))
        single.append(sg.half()); mats.append(F.pad(mt, (0, 0, 0, K - mt.size(1))).half())
        masks.append(F.pad(mk.float(), (0, K - mk.size(1))).bool())
    return torch.cat(single), torch.cat(mats), torch.cat(masks)


@torch.no_grad()
def split_scores(rt, split, cfg, reps=None):
    """-> numpy [nq, nc] scores for the split, plus the query / pool representations"""
    if reps is None:
        t0 = time.time()
        q = encode_docs(rt, [(split, "q", i) for i in SPL[split]["qids"]], cfg.k_eval, cfg.chunk_strategy)
        c = encode_docs(rt, [(split, "c", i) for i in SPL[split]["pool_ids"]], cfg.k_eval, cfg.chunk_strategy)
        enc_s = time.time() - t0
    else:
        (q, c), enc_s = reps, 0.0
    S = score_reps(tuple(x.float() if x.dtype == torch.float16 else x for x in q),
                   tuple(x.float() if x.dtype == torch.float16 else x for x in c), cfg.score_mode)
    return S.float().cpu().numpy(), (q, c), enc_s

In [ ]:

# ============================================================
# CONTRASTIVE TRAINING with GradCache (Gao et al., 2021)
#
# One step = B training queries, each with 1 relevant candidate + n_hard hard negatives; every other candidate in the
# batch is an in-batch negative (candidates relevant to the query are masked, never used as negatives).
# Long cases make each "example" K_train x 512 tokens, so a 40-document batch is ~160 chunk forwards. GradCache keeps
# the large contrastive batch on a T4:
#   1) encode every chunk WITHOUT a graph (RNG state saved per sub-batch)      -> chunk vectors V
#   2) loss(V) with the document aggregation; backward to V                  -> dL/dV (and grads of the aggregator)
#   3) re-encode each sub-batch WITH a graph under the same RNG state and back-propagate dL/dV through it
# The gradient is exactly that of the full batch, with activation memory of one sub-batch.
#
# Objectives: "infonce" (softmax over the batch, Oord et al. 2018) | "pairwise" (logistic loss on every
# positive-negative pair, RankNet-style). KD (Hinton et al., 2015): listwise KL between the teacher's and the student's
# score distributions over the same batch + cosine alignment of student and teacher case vectors.
# ============================================================
def hard_negative_table(S, split, depth):
    """top-`depth` non-relevant, allowed candidates per query from a score matrix (BM25, teacher or the model)"""
    S = np.where(valid_mask(split), S, -np.inf)
    pids, rel = SPL[split]["pool_ids"], SPL[split]["rel"]
    order = np.argsort(-S, axis=1, kind="stable")[:, : depth + 50]
    table = {}
    for i, q in enumerate(SPL[split]["qids"]):
        table[q] = [pids[j] for j in order[i] if np.isfinite(S[i, j]) and pids[j] not in rel[q]][:depth]
    return table


def make_batch(qids, hn, cfg, rng):
    rel = SPL["train"]["rel"]
    allowed = SPL["train"].get("allowed")
    pool_ids = SPL["train"]["pool_ids"]
    cands, pos_col = [], []
    for q in qids:
        pos = str(rng.choice(sorted(rel[q])))
        pos_col.append(len(cands)); cands.append(pos)
        neg_src = hn.get(q) if hn else None
        if not neg_src:
            src = allowed[q] if allowed else pool_ids
            neg_src = [c for c in rng.choice(src, min(len(src), 4 * cfg.n_hard + 4), replace=False)
                       if c not in rel[q] and c != q]
        k = min(cfg.n_hard, len(neg_src))
        cands += [str(c) for c in rng.choice(neg_src[: cfg.hn_depth], k, replace=False)] if k else []
    B, Nc = len(qids), len(cands)
    invalid = torch.zeros(B, Nc, dtype=torch.bool)
    for i, q in enumerate(qids):
        ok = set(allowed[q]) if allowed else None
        for j, c in enumerate(cands):
            if j != pos_col[i] and (c in rel[q] or c == q or (ok is not None and c not in ok)):
                invalid[i, j] = True
    return cands, torch.tensor(pos_col), invalid


def batch_loss(rt, V, doc, slot, B, pos_col, invalid, cfg, teacher_T=None, teacher_rep=None):
    n_docs = int(doc.max()) + 1
    single, mat, msk = rt.doc_reps(V, doc, slot, n_docs)
    q = (single[:B], mat[:B], msk[:B]); c = (single[B:], mat[B:], msk[B:])
    S = score_reps(q, c, cfg.score_mode) * rt.scale()
    inv = invalid.to(S.device)
    S = S.masked_fill(inv, -1e4)
    pc = pos_col.to(S.device)
    if cfg.objective == "infonce":
        loss = F.cross_entropy(S, pc)
    elif cfg.objective == "pairwise":
        sp_ = S.gather(1, pc[:, None])
        neg = ~inv; neg[torch.arange(B), pc] = False
        loss = F.softplus(S - sp_)[neg].mean()
    else:
        raise ValueError(cfg.objective)
    if teacher_T is not None:
        T = teacher_T.to(S.device).masked_fill(inv, -1e4)
        t = cfg.kd_temp
        kl = F.kl_div(F.log_softmax(S / t, -1), F.log_softmax(T / t, -1), log_target=True, reduction="batchmean") * t * t
        loss = (1 - cfg.kd_alpha) * loss + cfg.kd_alpha * kl
    if teacher_rep is not None and cfg.kd_rep > 0:
        loss = loss + cfg.kd_rep * (1 - (single * teacher_rep.to(single.device)).sum(-1)).mean()
    return loss


def gradcache_step(rt, flat, doc, slot, loss_fn, scaler, cfg):
    sbs = sub_batches(flat, cfg.enc_bs, cfg.enc_tok_budget)
    V = torch.empty(len(flat), D_MODEL, device=device)
    states = []
    with torch.no_grad():
        for sb in sbs:
            states.append(get_rng())
            ids, am = pad_batch([flat[i] for i in sb])
            with amp_ctx():
                V[torch.tensor(sb, device=device)] = rt.chunk_vec(ids, am).float()
    V.requires_grad_(True)
    loss = loss_fn(V, doc, slot)
    scaler.scale(loss).backward()
    G = V.grad.detach()
    if not torch.isfinite(G).all():
        return float("nan")
    for sb, st in zip(sbs, states):
        set_rng(st)
        ids, am = pad_batch([flat[i] for i in sb])
        idx = torch.tensor(sb, device=device)
        with amp_ctx():
            v = rt.chunk_vec(ids, am).float()
        v.backward(G[idx])
    return float(loss)


def train_retriever(rt, cfg, tag, teacher_cache=None):
    rng = np.random.RandomState(cfg.seed)
    pre, head = rt.pretrained_ids, rt.head_param_ids()
    groups = []
    for name, sel, lr in (("head", lambda p: id(p) in head, cfg.lr_head),
                          ("new", lambda p: id(p) not in head and id(p) not in pre, cfg.lr),
                          ("backbone", lambda p: id(p) in pre, cfg.lr_backbone)):
        ps = [p for p in rt.parameters() if p.requires_grad and sel(p)]
        if ps:
            groups.append(dict(params=ps, lr=lr, name=name))
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd)
    train_q = [q for q in SPL["train"]["qids"] if SPL["train"]["rel"][q]][: cfg.max_train_q or None]
    steps_ep = math.ceil(len(train_q) / cfg.batch_q)
    total = steps_ep * cfg.epochs; warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

    if cfg.hn_source == "teacher":
        assert teacher_cache is not None, "hn_source='teacher' needs a teacher"
        hn = hard_negative_table(teacher_cache["S"], "train", cfg.hn_depth)
    elif cfg.hn_source == "bm25":
        hn = hard_negative_table(bm25_scores("train"), "train", cfg.hn_depth)
    elif cfg.hn_source == "dense":                              # the model's own (zero-shot at start) ranking
        hn = hard_negative_table(split_scores(rt, "train", cfg)[0], "train", cfg.hn_depth)
    else:
        hn = None
    t_idx = None
    if teacher_cache is not None:
        t_idx = {("q", k): i for i, k in enumerate(SPL["train"]["qids"])}
        t_idx.update({("c", k): i for i, k in enumerate(SPL["train"]["pool_ids"])})

    best, bad = -1.0, 0
    ck_path = os.path.join(local_ck, f"{tag}.pt")
    print(f"[{tag}] {len(train_q)} train queries | {steps_ep} steps/epoch x {cfg.epochs} | batch {cfg.batch_q} queries "
          f"x (1 pos + {cfg.n_hard} hard neg, hn={cfg.hn_source}) | K_train {cfg.k_train} chunks | "
          + " ".join(f"{g['name']} {sum(p.numel() for p in g['params'])/1e6:.2f}M" for g in groups))
    for ep in range(cfg.epochs):
        rt.train(); t0, run, n = time.time(), 0.0, 0
        perm = rng.permutation(len(train_q))
        for st in range(steps_ep):
            qids = [train_q[i] for i in perm[st * cfg.batch_q:(st + 1) * cfg.batch_q]]
            if not qids:
                continue
            cands, pos_col, invalid = make_batch(qids, hn, cfg, rng)
            keys = [("train", "q", q) for q in qids] + [("train", "c", c) for c in cands]
            flat, doc, slot = layout([doc_chunk_list(TOK[k], cfg.k_train, cfg.chunk_strategy, rng) for k in keys])
            tT = tR = None
            if teacher_cache is not None:
                qi = torch.tensor([t_idx[("q", q)] for q in qids]); ci = torch.tensor([t_idx[("c", c)] for c in cands])
                tq = tuple(x[qi.to(x.device)].float() for x in teacher_cache["q"])
                tc = tuple(x[ci.to(x.device)].float() for x in teacher_cache["c"])
                tT = score_reps(tq, tc, teacher_cache["mode"]) * teacher_cache["scale"]
                tR = torch.cat([tq[0], tc[0]], 0)
            fn = lambda V, d, s: batch_loss(rt, V, d, s, len(qids), pos_col, invalid, cfg, tT, tR)
            loss = gradcache_step(rt, flat, doc, slot, fn, scaler, cfg)
            if not math.isfinite(loss):                         # fp16 overflow in dL/dV: skip, halve the loss scale
                print(f"[{tag}] non-finite step skipped"); opt.zero_grad(set_to_none=True)
                if scaler.is_enabled():
                    scaler.update(scaler.get_scale() / 2)
                continue
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            run += loss; n += 1
            if (st + 1) % cfg.log_every == 0:
                print(f"[{tag}] ep {ep+1} step {st+1}/{steps_ep} loss {run/n:.4f} scale {float(rt.scale()):.1f} "
                      f"({time.time()-t0:.0f}s)")
        S, _, _ = split_scores(rt, "dev", cfg)
        m = full_metrics(S, "dev")
        print(f"[{tag}] ep {ep+1} train loss {run/max(n,1):.4f} | DEV {fmt_m(m)} ({time.time()-t0:.0f}s)")
        if m["f1"] > best:
            best, bad = m["f1"], 0
            save_trainable(rt, ck_path, {"cfg": vars(cfg), "epoch": ep + 1, "dev": m})
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
        if cfg.hn_refresh and ep + 1 < cfg.epochs:              # ANCE-style: re-mine with the current model
            hn = hard_negative_table(split_scores(rt, "train", cfg)[0], "train", cfg.hn_depth)
            print(f"[{tag}] hard negatives re-mined with the current model")
    load_trainable(rt, ck_path)
    shutil.copy(ck_path, os.path.join(p7_dir, f"ckpt_{tag}.pt"))
    return best


@torch.no_grad()
def build_teacher_cache(teacher, tcfg):
    """teacher representations of every train query / candidate + the full train score matrix (for KD + hard negatives)"""
    S, (q, c), _ = split_scores(teacher, "train", tcfg)
    return dict(S=S, q=q, c=c, mode=tcfg.score_mode, scale=float(teacher.scale()))

In [ ]:
# ============================================================
# CONFIG + EXPERIMENT RUNNER
# ============================================================
BASE_CFG = dict(
    method="lorma",
    # document representation
    k_train=4, k_eval=8, chunk_strategy="uniform", pool="attn", score_mode="single",
    # objective / negatives
    objective="infonce", tau=0.05, batch_q=8, n_hard=3, hn_source="bm25", hn_depth=50, hn_refresh=False,
    # adaptation hyper-parameters
    r=16, alpha=32, lora_dropout=0.05, prefix_len=16, prompt_len=20, unfreeze_top_k=3,
    # optimisation
    lr=5e-4, lr_head=1e-3, lr_backbone=2e-5, wd=0.01, epochs=3, patience=2, warmup=0.06, clip=1.0,
    grad_ckpt=True, enc_bs=16, enc_tok_budget=8192, seed=42, log_every=25, max_train_q=None,
    # knowledge distillation
    teacher_ckpt=None, kd_alpha=0.5, kd_temp=1.0, kd_rep=0.5, student_layers=None,
)
METHOD_DEFAULTS = {
    "zeroshot": dict(),
    "full":     dict(lr_backbone=2e-5),
    "partial":  dict(lr_backbone=5e-5),
    "lorma":    dict(lr=5e-4),
    "qlora":    dict(lr=5e-4),
    "lora":     dict(lr=5e-4),
    "ia3":      dict(lr=3e-3),
    "prefix":   dict(lr=3e-3),
    "prompt":   dict(lr=3e-3),
    "kd":       dict(lr_backbone=5e-5, hn_source="teacher"),
}


def make_cfg(**over):
    meth = over.get("method", BASE_CFG["method"])
    return SimpleNamespace(**{**BASE_CFG, **MODEL_CFG_OVERRIDES, **METHOD_DEFAULTS.get(meth, {}), **over})


RESULTS = {}
_res_path = os.path.join(p7_dir, "results.json")
if os.path.exists(_res_path):
    RESULTS = json.load(open(_res_path))


def build_retriever(cfg, backbone=None):
    """backbone=None: fresh copy of the Phase-1 model with cfg.method applied; else a (student) backbone."""
    if backbone is None:
        bb, desc, pre = build_backbone(cfg)          # pre = ids of the Phase-1 weights, collected BEFORE injection
    else:
        bb, desc = backbone, "student backbone (all weights trainable)"
        pre = {id(p) for p in bb.parameters()}
    rt = Retriever(bb, cfg).to(device)
    rt.pretrained_ids = pre                          # -> lr_backbone; PEFT parameters -> lr; head -> lr_head
    rt.method_desc = desc
    return rt


def _from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = SimpleNamespace(**ck["cfg"])
    if cfg.method == "kd":
        tch, _ = _from_ckpt(cfg.teacher_ckpt)
        rt = build_retriever(cfg, backbone=make_student_backbone(tch.bb, cfg)); del tch
    else:
        rt = build_retriever(cfg)
    load_trainable(rt, path)
    return rt, cfg


def evaluate_and_submit(rt, cfg, name, submit=True):
    S_dev, _, enc_dev = split_scores(rt, "dev", cfg)
    m = full_metrics(S_dev, "dev")
    (w, f1h), _ = tune_fusion(S_dev, "dev")
    nq = len(SPL["dev"]["qids"]); npool = len(SPL["dev"]["pool_ids"])
    print(f"[{name}] DEV dense : {fmt_m(m)}")
    print(f"[{name}] DEV hybrid: F1@7 {f1h:.4f} with w_dense={w:.1f} (BM25 alone = w 0.0)")
    RESULTS[name] = dict(dev=m, dev_hybrid=dict(f1=f1h, w=w), method=cfg.method, desc=rt.method_desc,
                         trainable=sum(p.numel() for p in rt.parameters() if p.requires_grad),
                         deployed=sum(p.numel() for p in rt.parameters()),
                         enc_sec_per_doc=enc_dev / max(nq + npool, 1),
                         cfg={k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))})
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    if submit:
        S_test, _, _ = split_scores(rt, "test", cfg)
        write_submission(S_test, name)
        if w < 1.0:
            write_submission(w * zrow(S_test) + (1 - w) * zrow(bm25_scores("test")), name + "_hybrid")
    return m


def run_experiment(name, submit=True, **over):
    cfg = make_cfg(**over)
    set_seed(cfg.seed)
    tcache = None
    if cfg.method == "kd":
        assert cfg.teacher_ckpt and os.path.exists(cfg.teacher_ckpt), "kd needs teacher_ckpt (e.g. ckpt of the full run)"
        teacher, tcfg = _from_ckpt(cfg.teacher_ckpt)
        for k in ("pool", "score_mode", "k_eval"):
            setattr(cfg, k, getattr(tcfg, k))
        print("caching teacher representations of the train split ...")
        tcache = build_teacher_cache(teacher, tcfg)
        rt = build_retriever(cfg, backbone=make_student_backbone(teacher.bb, cfg))
        rt.load_state_dict({k: v for k, v in teacher.state_dict().items() if not k.startswith("bb.")}, strict=False)
        del teacher; gc.collect(); torch.cuda.empty_cache()
    else:
        rt = build_retriever(cfg)
    print(f"\n=== {name} | {rt.method_desc} | trainable {sum(p.numel() for p in rt.parameters() if p.requires_grad)/1e6:.2f}M"
          f" / {sum(p.numel() for p in rt.parameters())/1e6:.1f}M ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    if cfg.method != "zeroshot":
        train_retriever(rt, cfg, name, teacher_cache=tcache)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    del tcache
    m = evaluate_and_submit(rt, cfg, name, submit)
    RESULTS[name].update(train_min=train_min, peak_gb=peak)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    del rt; gc.collect(); torch.cuda.empty_cache()
    return m


def bm25_reference(submit=False):
    """lexical reference point (not a Phase-1-model system; submit only if you want it on the leaderboard)"""
    m = full_metrics(bm25_scores("dev"), "dev")
    print("BM25 (BPE unigrams + bigrams) DEV:", fmt_m(m))
    RESULTS["BM25_reference"] = dict(dev=m, method="bm25", desc="BM25 on BPE ids", trainable=0, deployed=0)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    if submit:
        write_submission(bm25_scores("test"), "BM25_reference")
    return m


def show_results():
    print(f"\n{'experiment':24s} {'method':9s} {'F1@7':>7s} {'P@7':>7s} {'R@7':>7s} {'R@100':>6s} {'MRR':>6s} "
          f"{'hybrid':>7s} {'train M':>8s} {'model M':>8s} {'ms/doc':>7s} {'min':>5s} {'GB':>5s}")
    for n, r in RESULTS.items():
        d = r["dev"]
        print(f"{n:24s} {r['method']:9s} {d['f1']:7.4f} {d['p']:7.4f} {d['r']:7.4f} {d.get('r100', 0):6.3f} "
              f"{d.get('mrr', 0):6.3f} {r.get('dev_hybrid', {}).get('f1', 0):7.4f} {r['trainable']/1e6:8.2f} "
              f"{r['deployed']/1e6:8.1f} {1000 * r.get('enc_sec_per_doc', 0):7.1f} {r.get('train_min', 0):5.0f} "
              f"{r.get('peak_gb', 0):5.1f}")

In [ ]:
# ============================================================
# SELF-TESTS (a minute): run before training
#   1) the backbone patch reproduces the Phase-1 model
#   2) LoRMA / LoRA / IA3 leave the retriever unchanged at init (QLoRA: 4-bit base stays close)
#   3) GradCache gives the SAME gradients as ordinary back-propagation through the whole batch (dropout on)
# ============================================================
@torch.no_grad()
def check_peft_identity():
    keys = [("train", "c", i) for i in SPL["train"]["pool_ids"][:6]]
    ref = encode_docs(build_retriever(make_cfg(method="zeroshot")), keys, 2)[0].float()
    for meth in ("lorma", "lora", "ia3", "qlora"):
        rt = build_retriever(make_cfg(method=meth))
        out = encode_docs(rt, keys, 2)[0].float()
        cos = (out * ref).sum(-1).min().item()
        print(f"  {meth:6s} at init: min cosine to zero-shot case vectors {cos:.5f}" + ("  (4-bit base)" if meth == "qlora" else ""))
        del rt
    torch.cuda.empty_cache()


def check_gradcache():
    global AMP_DTYPE
    saved, AMP_DTYPE = AMP_DTYPE, None                        # fp32 for a tight comparison
    try:
        cfg = make_cfg(method="lorma", batch_q=2, n_hard=1, k_train=2, enc_bs=2)
        rt = build_retriever(cfg); rt.train()
        rng = np.random.RandomState(0)
        qids = [q for q in SPL["train"]["qids"] if SPL["train"]["rel"][q]][:2]
        cands, pos, inv = make_batch(qids, None, cfg, rng)
        keys = [("train", "q", q) for q in qids] + [("train", "c", c) for c in cands]
        flat, doc, slot = layout([doc_chunk_list(TOK[k], cfg.k_train, cfg.chunk_strategy, rng) for k in keys])
        for i in range(len(flat)):
            flat[i] = flat[i][:128]                           # short chunks keep the test fast
        fn = lambda V, d, s: batch_loss(rt, V, d, s, len(qids), pos, inv, cfg)
        params = [(n, p) for n, p in rt.named_parameters() if p.requires_grad]
        st0 = get_rng()
        l1 = gradcache_step(rt, flat, doc, slot, fn, torch.amp.GradScaler("cuda", enabled=False), cfg)
        g1 = {n: p.grad.detach().clone() for n, p in params if p.grad is not None}
        rt.zero_grad(set_to_none=True)
        set_rng(st0)
        V = torch.zeros(len(flat), D_MODEL, device=device)
        for sb in sub_batches(flat, cfg.enc_bs, cfg.enc_tok_budget):
            ids, am = pad_batch([flat[i] for i in sb])
            V = V.index_put((torch.tensor(sb, device=device),), rt.chunk_vec(ids, am).float())
        l2 = fn(V, doc, slot); l2.backward()
        worst = 0.0
        for n, p in params:
            if p.grad is None or n not in g1:
                continue
            worst = max(worst, ((g1[n] - p.grad).abs().max() / (p.grad.abs().max() + 1e-8)).item())
        print(f"GradCache vs direct backprop: loss {l1:.5f} vs {float(l2):.5f} | worst relative grad diff {worst:.2e}")
        assert abs(l1 - float(l2)) < 1e-4 and worst < 1e-2, "GradCache gradients differ from direct back-propagation"
        del rt; torch.cuda.empty_cache()
    finally:
        AMP_DTYPE = saved


check_backbone()
check_peft_identity()
check_gradcache()

lean vs Phase-1 attention (no memory): max|diff| 1.19e-06
lean vs Phase-1 attention (with 16 memory slots): max|diff| 1.49e-06
torch.float16 autocast: finite=True, min cosine to fp32 0.5422
-> mixed precision switched OFF (fp32 everywhere)
  lorma  at init: min cosine to zero-shot case vectors 0.99996
  lora   at init: min cosine to zero-shot case vectors 0.99996
  ia3    at init: min cosine to zero-shot case vectors 0.99996
  qlora  at init: min cosine to zero-shot case vectors 0.99992  (4-bit base)


/tmp/ipykernel_656/1427953498.py:102: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  return float(loss)


GradCache vs direct backprop: loss 1.38653 vs 1.38653 | worst relative grad diff 2.84e-05


In [ ]:
# ============================================================
# 5. RUN  (dev Micro-F1@7 is printed after every epoch; every run writes submission_<name>/submission.zip, plus a
#          _hybrid variant (dense + BM25 fusion, weight tuned on dev) when fusion helps)
# ============================================================
# (0) Reference points (minutes, no training): lexical BM25 and the zero-shot Phase-1 GQLA
bm25_reference()
run_experiment("X0_zeroshot", method="zeroshot", submit=False)
run_experiment("X_smoke", method="lorma", max_train_q=32, epochs=1, submit=False)   # whole pipeline, a few minutes

# (1) Six substantially different techniques from the Phase-7 list (LoRMA compulsory, KD mandatory).
#     Same representation (8 chunks, attention pooling, single vector), same objective and negatives,
#     so the differences come from the adaptation technique.
# run_experiment("X1_full",    method="full", batch_q=8)                 # strongest; the KD teacher
# run_experiment("X2_lorma",   method="lorma")                           # compulsory
# run_experiment("X3_ia3",     method="ia3")
# run_experiment("X4_prefix",  method="prefix")
# run_experiment("X5_partial", method="partial", unfreeze_top_k=3)
# run_experiment("X6_kd",      method="kd", teacher_ckpt=os.path.join(p7_dir, "ckpt_X1_full.pt"), student_layers=4)   # layer-strided blocks [0, 2, 5, 7]

# (2) Extras: other techniques, and representation / objective / negative ablations (not separate techniques)
# run_experiment("X7_qlora",    method="qlora")
# run_experiment("X8_prompt",   method="prompt")
# run_experiment("X9_maxsim",   method="lorma", score_mode="maxsim")     # chunk-level late interaction
# run_experiment("X10_last",    method="lorma", pool="last")             # last-token (</s>) pooling, as in Phase 4
# run_experiment("X11_pair",    method="lorma", objective="pairwise")
# run_experiment("X12_randneg", method="lorma", hn_source="none")        # in-batch + random negatives only
# run_experiment("X13_ance",    method="lorma", hn_source="dense", hn_refresh=True)
# run_experiment("X14_head",    method="lorma", chunk_strategy="head")

show_results()

BM25 (BPE unigrams + bigrams) DEV: F1@7 0.2023 (P 0.2203 R 0.1871) | R@50 0.468 R@100 0.582 MRR 0.586

=== X0_zeroshot | Phase-1 Transformer-XL as is (no training) | trainable 0.59M / 76.6M ===
[X0_zeroshot] DEV dense : F1@7 0.0111 (P 0.0121 R 0.0103) | R@50 0.102 R@100 0.168 MRR 0.049
[X0_zeroshot] DEV hybrid: F1@7 0.2034 with w_dense=0.1 (BM25 alone = w 0.0)

=== X_smoke | LoRMA+ r=16 on 48 linears | trainable 2.07M / 78.0M ===
[X_smoke] 32 train queries | 4 steps/epoch x 1 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.59M new 1.47M
[X_smoke] ep 1 train loss 3.4627 | DEV F1@7 0.0511 (P 0.0557 R 0.0473) | R@50 0.176 R@100 0.255 MRR 0.217 (445s)
[X_smoke] DEV dense : F1@7 0.0511 (P 0.0557 R 0.0473) | R@50 0.176 R@100 0.255 MRR 0.217
[X_smoke] DEV hybrid: F1@7 0.2034 with w_dense=0.1 (BM25 alone = w 0.0)

experiment               method       F1@7     P@7     R@7  R@100    MRR  hybrid  train M  model M  ms/doc   min    GB
BM25_reference           bm25    

In [ ]:
run_experiment("X1_full",    method="full", batch_q=8)                 # strongest; the KD teacher



=== X1_full | all Transformer-XL weights | trainable 76.55M / 76.6M ===
[X1_full] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.59M backbone 75.96M
[X1_full] ep 1 step 25/103 loss 3.4609 scale 19.9 (807s)
[X1_full] ep 1 step 50/103 loss 3.4628 scale 19.9 (1610s)
[X1_full] ep 1 step 75/103 loss 3.4575 scale 20.2 (2416s)
[X1_full] ep 1 step 100/103 loss 3.4277 scale 20.3 (3218s)
[X1_full] ep 1 train loss 3.4032 | DEV F1@7 0.0178 (P 0.0194 R 0.0164) | R@50 0.109 R@100 0.183 MRR 0.062 (3570s)
[X1_full] ep 2 step 25/103 loss 3.2656 scale 20.4 (804s)
[X1_full] ep 2 step 50/103 loss 3.2172 scale 20.4 (1609s)
[X1_full] ep 2 step 75/103 loss 3.1975 scale 20.4 (2415s)
[X1_full] ep 2 step 100/103 loss 3.1722 scale 20.4 (3212s)
[X1_full] ep 2 train loss 3.1509 | DEV F1@7 0.0222 (P 0.0242 R 0.0206) | R@50 0.137 R@100 0.221 MRR 0.103 (3563s)
[X1_full] ep 3 step 25/103 loss 3.0715 scale 20.4 (806s)
[X1_full] ep 3 step 50/103 los

{'f1': 0.03001667593107282,
 'p': 0.03268765133171913,
 'r': 0.02774922918807811,
 'r50': 0.12328083102992926,
 'r100': 0.22328376050735857,
 'mrr': 0.12043392938950806}

In [ ]:
run_experiment("X2_lorma",   method="lorma")                           # compulsory



=== X2_lorma | LoRMA+ r=16 on 48 linears | trainable 2.07M / 78.0M ===
[X2_lorma] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.59M new 1.47M
[X2_lorma] ep 1 step 25/103 loss 3.4608 scale 19.9 (819s)
[X2_lorma] ep 1 step 50/103 loss 3.4563 scale 20.1 (1633s)
[X2_lorma] ep 1 step 75/103 loss 3.4592 scale 20.0 (2448s)
[X2_lorma] ep 1 step 100/103 loss 3.4602 scale 20.0 (3258s)
[X2_lorma] ep 1 train loss 3.4401 | DEV F1@7 0.0022 (P 0.0024 R 0.0021) | R@50 0.052 R@100 0.090 MRR 0.034 (3633s)
[X2_lorma] ep 2 step 25/103 loss 3.4637 scale 20.0 (811s)
[X2_lorma] ep 2 step 50/103 loss 3.4636 scale 20.0 (1621s)
[X2_lorma] ep 2 step 75/103 loss 3.4633 scale 20.0 (2435s)
[X2_lorma] ep 2 step 100/103 loss 3.4633 scale 20.0 (3242s)
[X2_lorma] ep 2 train loss 3.4432 | DEV F1@7 0.0078 (P 0.0085 R 0.0072) | R@50 0.049 R@100 0.093 MRR 0.034 (3616s)
[X2_lorma] ep 3 step 25/103 loss 3.4640 scale 20.0 (811s)
[X2_lorma] ep 3 step 50/1

{'f1': 0.008893829905503055,
 'p': 0.009685230024213076,
 'r': 0.008221993833504625,
 'r50': 0.057226644810469435,
 'r100': 0.10363378892967184,
 'mrr': 0.03525191461898377}

In [ ]:
run_experiment("X3_ia3",     method="ia3")



=== X3_ia3 | IA3 on k_proj, v_proj and FFN hidden: 24 vectors | trainable 0.62M / 76.6M ===
[X3_ia3] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.59M new 0.03M
[X3_ia3] ep 1 step 25/103 loss 3.4609 scale 19.9 (735s)
[X3_ia3] ep 1 step 50/103 loss 3.4628 scale 19.9 (1467s)
[X3_ia3] ep 1 step 75/103 loss 3.4635 scale 19.9 (2201s)
[X3_ia3] ep 1 step 100/103 loss 3.4634 scale 19.9 (2931s)
[X3_ia3] ep 1 train loss 3.4433 | DEV F1@7 0.0133 (P 0.0145 R 0.0123) | R@50 0.068 R@100 0.145 MRR 0.051 (3280s)
[X3_ia3] ep 2 step 25/103 loss 3.4630 scale 20.1 (732s)
[X3_ia3] ep 2 step 50/103 loss 3.4627 scale 20.2 (1464s)
[X3_ia3] ep 2 step 75/103 loss 3.4626 scale 20.2 (2198s)
[X3_ia3] ep 2 step 100/103 loss 3.4628 scale 20.2 (2924s)
[X3_ia3] ep 2 train loss 3.4427 | DEV F1@7 0.0089 (P 0.0097 R 0.0082) | R@50 0.081 R@100 0.144 MRR 0.054 (3273s)
[X3_ia3] ep 3 step 25/103 loss 3.4639 scale 20.2 (733s)
[X3_ia3] ep 3 step 50/103 lo

{'f1': 0.013340744858254585,
 'p': 0.014527845036319613,
 'r': 0.012332990750256937,
 'r50': 0.0676781420042398,
 'r100': 0.14453153391633433,
 'mrr': 0.05078251701057313}

In [ ]:
run_experiment("X4_prefix",  method="prefix")



=== X4_prefix | prefix tuning: 16 learned memory slots per block (Transformer-XL memory, real-state init) | trainable 0.69M / 76.7M ===
[X4_prefix] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.59M new 0.10M
[X4_prefix] ep 1 step 25/103 loss 3.4608 scale 19.9 (735s)
[X4_prefix] ep 1 step 50/103 loss 3.4628 scale 20.0 (1467s)
[X4_prefix] ep 1 step 75/103 loss 3.4634 scale 20.0 (2201s)
[X4_prefix] ep 1 step 100/103 loss 3.4630 scale 20.3 (2931s)
[X4_prefix] ep 1 train loss 3.4428 | DEV F1@7 0.0189 (P 0.0206 R 0.0175) | R@50 0.114 R@100 0.198 MRR 0.071 (3287s)
[X4_prefix] ep 2 step 25/103 loss 3.4628 scale 20.4 (732s)
[X4_prefix] ep 2 step 50/103 loss 3.4621 scale 20.5 (1464s)
[X4_prefix] ep 2 step 75/103 loss 3.4603 scale 20.7 (2198s)
[X4_prefix] ep 2 step 100/103 loss 3.4696 scale 20.9 (2927s)
[X4_prefix] ep 2 train loss 3.4492 | DEV F1@7 0.0211 (P 0.0230 R 0.0195) | R@50 0.122 R@100 0.220 MRR 0.066 (3283s)
[X4_pre

{'f1': 0.021122846025569762,
 'p': 0.023002421307506054,
 'r': 0.019527235354573486,
 'r50': 0.12172795323193952,
 'r100': 0.22034822895383369,
 'mrr': 0.06605782719149686}

In [ ]:
run_experiment("X5_partial", method="partial", unfreeze_top_k=3)



=== X5_partial | top 3 blocks + final norm | trainable 19.64M / 76.6M ===
[X5_partial] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.59M backbone 19.05M
[X5_partial] ep 1 step 25/103 loss 3.4609 scale 19.9 (480s)
[X5_partial] ep 1 step 50/103 loss 3.4628 scale 19.9 (971s)
[X5_partial] ep 1 step 75/103 loss 3.4635 scale 19.9 (1463s)
[X5_partial] ep 1 step 100/103 loss 3.4634 scale 20.0 (1952s)
[X5_partial] ep 1 train loss 3.4431 | DEV F1@7 0.0233 (P 0.0254 R 0.0216) | R@50 0.099 R@100 0.177 MRR 0.086 (2267s)
[X5_partial] ep 2 step 25/103 loss 3.4621 scale 20.1 (491s)
[X5_partial] ep 2 step 50/103 loss 3.4645 scale 20.2 (981s)
[X5_partial] ep 2 step 75/103 loss 3.4622 scale 20.3 (1474s)
[X5_partial] ep 2 step 100/103 loss 3.4470 scale 20.3 (1962s)
[X5_partial] ep 2 train loss 3.4250 | DEV F1@7 0.0189 (P 0.0206 R 0.0175) | R@50 0.071 R@100 0.134 MRR 0.056 (2278s)
[X5_partial] ep 3 step 25/103 loss 3.4279 scale 20.2 (

In [ ]:
run_experiment("X6_kd",      method="kd", teacher_ckpt=os.path.join(p7_dir, "ckpt_X1_full.pt"), student_layers=4)   # layer-strided blocks [0, 2, 5, 7]

caching teacher representations of the train split ...
student: 4 blocks from teacher blocks [0, 2, 5, 7] | 50.6M params (teacher 76.0M)

=== X6_kd | student backbone (all weights trainable) | trainable 51.16M / 51.2M ===
[X6_kd] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=teacher) | K_train 4 chunks | head 0.59M backbone 50.56M
[X6_kd] ep 1 step 25/103 loss 2.0056 scale 20.5 (391s)
[X6_kd] ep 1 step 50/103 loss 1.8625 scale 20.5 (775s)
[X6_kd] ep 1 step 75/103 loss 1.8026 scale 20.5 (1164s)
[X6_kd] ep 1 step 100/103 loss 1.7761 scale 20.5 (1548s)
[X6_kd] ep 1 train loss 1.7609 | DEV F1@7 0.0189 (P 0.0206 R 0.0175) | R@50 0.128 R@100 0.287 MRR 0.074 (1722s)
[X6_kd] ep 2 step 25/103 loss 1.5869 scale 20.5 (387s)
[X6_kd] ep 2 step 50/103 loss 1.6141 scale 20.5 (776s)
[X6_kd] ep 2 step 75/103 loss 1.6109 scale 20.5 (1162s)
[X6_kd] ep 2 step 100/103 loss 1.6196 scale 20.5 (1554s)
[X6_kd] ep 2 train loss 1.6163 | DEV F1@7 0.0256 (P 0.0278 R 0.0236) | 

{'f1': 0.02556976097832129,
 'p': 0.02784503631961259,
 'r': 0.023638232271325797,
 'r50': 0.13540687855263572,
 'r100': 0.2823305877106402,
 'mrr': 0.08989257297970969}

In [ ]:
show_results()